# Final HNSW Notebook

This notebook rebuilds a handwritten HNSW index from scratch using Python + NumPy only. Exact search remains the ground truth. The implementation is built for correctness first and then benchmarked on 1K and 10K subsets before any larger-scale run.

In [15]:
import json
import time
import heapq
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


In [16]:
PROJECT_DIR = Path("..")
EMBEDDINGS_FILE = PROJECT_DIR / "data" / "embeddings" / "embeddings.npy"
IDS_FILE = PROJECT_DIR / "data" / "embeddings" / "ids.npy"

embeddings = np.load(EMBEDDINGS_FILE)
ids = np.load(IDS_FILE).astype(np.int64)
print(f"Vectors: {embeddings.shape[0]}")
print(f"Dimensions: {embeddings.shape[1]}")
print(f"ID count: {ids.shape[0]}")


Vectors: 119921
Dimensions: 384
ID count: 119921


In [17]:
def cosine_similarity(a, b):
    return float(np.dot(a, b))


def exact_search(query_vector, vectors, vector_ids, k=10):
    scores = vectors @ query_vector
    top_idx = np.argsort(scores)[::-1][:k]
    return [
        {
            "id": int(vector_ids[i]),
            "score": float(scores[i]),
            "index": int(i),
        }
        for i in top_idx
    ]


def recall_at_k(exact_results, approx_results, k=10):
    exact_ids = {r["id"] for r in exact_results[:k]}
    approx_ids = {r["id"] for r in approx_results[:k]}
    if not exact_ids:
        return 0.0
    return len(exact_ids & approx_ids) / len(exact_ids)

print("Exact-search helpers ready.")


Exact-search helpers ready.


In [18]:
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

def embed_text(text):
    vector = model.encode(
        [text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0]
    return np.asarray(vector, dtype=np.float32)

print(f"Model loaded: {MODEL_NAME}")
print(f"Sample embedding norm: {np.linalg.norm(embed_text('technology and markets continue to grow')):.4f}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6488.34it/s]


Model loaded: all-MiniLM-L6-v2
Sample embedding norm: 1.0000


In [19]:
class HNSWIndex:
    def __init__(self, M=8, ef_construction=50, ef_search=20, seed=42):
        self.M = int(M)
        self.ef_construction = int(ef_construction)
        self.ef_search = int(ef_search)
        self.rng = np.random.default_rng(seed)

        self.vectors = []
        self.ids = []
        self.id_to_index = {}
        self.deleted = set()
        self.graph = []
        self.entry_point = None
        self.max_level = -1

    def __len__(self):
        return len(self.vectors)

    def random_level(self):
        level = 0
        while self.rng.random() < 0.5:
            level += 1
        return level

    def _search_layer(self, query_vector, entry_points, layer, ef):
        if not entry_points:
            return []

        visited = set(entry_points)
        candidates = []
        results = []

        for node in entry_points:
            score = cosine_similarity(query_vector, self.vectors[node])
            candidates.append((score, node))
            results.append((score, node))

        while candidates:
            current_score, current = max(candidates, key=lambda x: x[0])
            candidates.remove((current_score, current))

            worst_result_score = min(results, key=lambda x: x[0])[0] if len(results) >= ef else float("-inf")
            if len(results) >= ef and current_score < worst_result_score:
                break

            for neighbor in self.graph[layer].get(current, []):
                if neighbor in visited or neighbor in self.deleted:
                    continue
                visited.add(neighbor)
                score = cosine_similarity(query_vector, self.vectors[neighbor])

                if len(results) < ef or score > min(results, key=lambda x: x[0])[0]:
                    candidates.append((score, neighbor))
                    results.append((score, neighbor))

                    if len(results) > ef:
                        worst_score, worst_node = min(results, key=lambda x: x[0])
                        results.remove((worst_score, worst_node))

        return sorted(results, key=lambda x: x[0], reverse=True)

    def _select_neighbors(self, candidates, M):
        ordered = sorted(candidates, key=lambda x: x[0], reverse=True)
        unique = []
        seen = set()
        for _, node in ordered:
            if node in seen:
                continue
            seen.add(node)
            unique.append(node)
        return unique[:M]

    def _connect(self, node_a, node_b, layer):
        self.graph[layer].setdefault(node_a, [])
        self.graph[layer].setdefault(node_b, [])

        if node_b not in self.graph[layer][node_a]:
            self.graph[layer][node_a].append(node_b)
        if node_a not in self.graph[layer][node_b]:
            self.graph[layer][node_b].append(node_a)

        for node in (node_a, node_b):
            neighbors = self.graph[layer].get(node, [])
            if len(neighbors) <= self.M:
                continue
            ranked = sorted(
                neighbors,
                key=lambda n: cosine_similarity(self.vectors[node], self.vectors[n]),
                reverse=True,
            )
            self.graph[layer][node] = ranked[: self.M]

    def insert(self, vector, doc_id):
        vector = np.asarray(vector, dtype=np.float32)
        node_index = len(self.vectors)

        self.vectors.append(vector)
        self.ids.append(int(doc_id))
        self.id_to_index[int(doc_id)] = node_index

        level = self.random_level()
        while len(self.graph) <= level:
            self.graph.append({})
        for layer in range(level + 1):
            self.graph[layer].setdefault(node_index, [])

        if self.entry_point is None:
            self.entry_point = node_index
            self.max_level = level
            return node_index

        current = self.entry_point
        for layer in range(self.max_level, level, -1):
            nearest = self._search_layer(vector, [current], layer, 1)
            if nearest:
                current = nearest[0][1]

        for layer in range(min(self.max_level, level), -1, -1):
            candidates = self._search_layer(vector, [current], layer, self.ef_construction)
            neighbors = self._select_neighbors(candidates, self.M)
            for neighbor in neighbors:
                self._connect(node_index, neighbor, layer)
            if candidates:
                current = max(candidates, key=lambda x: x[0])[1]

        if level > self.max_level:
            self.entry_point = node_index
            self.max_level = level

        return node_index

    def search(self, query_vector, k=10):
        if self.entry_point is None:
            return []

        current = self.entry_point
        for layer in range(self.max_level, 0, -1):
            nearest = self._search_layer(query_vector, [current], layer, 1)
            if nearest:
                current = nearest[0][1]

        results = self._search_layer(query_vector, [current], 0, max(self.ef_search, k))
        results = sorted(results, key=lambda x: x[0], reverse=True)[:k]

        return [
            {
                "id": int(self.ids[node]),
                "score": float(score),
                "index": int(node),
            }
            for score, node in results
        ]

    def delete(self, doc_id):
        node = self.id_to_index.get(int(doc_id))
        if node is None:
            return False
        self.deleted.add(node)
        for layer in range(len(self.graph)):
            if node in self.graph[layer]:
                self.graph[layer].pop(node, None)
            for neighbor in list(self.graph[layer].keys()):
                if node in self.graph[layer].get(neighbor, []):
                    self.graph[layer][neighbor].remove(node)
        self.id_to_index.pop(int(doc_id), None)
        return True

    def validate_graph(self):
        active_nodes = sorted(set(range(len(self.vectors))) - self.deleted)
        if not active_nodes:
            return {
                "valid": True,
                "node_count": 0,
                "active_nodes": 0,
                "entry_point": None,
                "layers": len(self.graph),
                "avg_neighbors": 0.0,
                "max_neighbors": 0,
                "issues": [],
            }

        issues = []
        degrees = []

        for layer, layer_map in enumerate(self.graph):
            for node, neighbors in layer_map.items():
                if node in self.deleted:
                    continue
                if node in neighbors:
                    issues.append(f"Self-loop found at node {node} in layer {layer}")
                unique_neighbors = list(dict.fromkeys(neighbors))
                if len(unique_neighbors) != len(neighbors):
                    issues.append(f"Duplicate edges found at node {node} in layer {layer}")
                if any(n in self.deleted for n in neighbors):
                    issues.append(f"Deleted node appears as neighbor of node {node} in layer {layer}")
                degrees.append(len(unique_neighbors))

        if self.entry_point is None or self.entry_point in self.deleted:
            issues.append("Entry point is invalid or deleted.")

        visited = set()
        stack = [self.entry_point] if self.entry_point is not None else []
        while stack:
            current = stack.pop()
            if current in visited or current in self.deleted:
                continue
            visited.add(current)
            for neighbor in self.graph[0].get(current, []):
                if neighbor not in visited and neighbor not in self.deleted:
                    stack.append(neighbor)

        missing = sorted(set(active_nodes) - visited)
        if missing:
            issues.append(f"Layer-0 reachability missing for nodes: {missing[:10]}")

        return {
            "valid": len(issues) == 0,
            "node_count": len(self.vectors),
            "active_nodes": len(active_nodes),
            "entry_point": self.entry_point,
            "layers": len(self.graph),
            "avg_neighbors": float(np.mean(degrees)) if degrees else 0.0,
            "max_neighbors": max(degrees) if degrees else 0,
            "issues": issues,
        }

print("Handwritten HNSW implementation ready.")


Handwritten HNSW implementation ready.


In [20]:
# Graph diagnostics: explain exactly why validation returns False.
# This checks the actual structure without changing HNSW parameters or running a new benchmark.

def diagnose_hnsw_graph(index, max_examples=10):
    total_nodes = len(index.vectors)
    print("=== HNSW GRAPH DIAGNOSTIC ===")
    print(f"total_nodes={total_nodes}")
    print(f"entry_point={index.entry_point}")
    print(f"max_level={index.max_level}")
    print(f"graph_layers={len(index.graph)}")
    print(f"deleted_nodes={len(index.deleted)}")

    layer_counts = []
    for layer in range(len(index.graph)):
        count = len(index.graph[layer])
        layer_counts.append(count)
        print(f"layer {layer}: node_count={count}")
    print("layer_counts:", layer_counts)

    issues = []
    invalid_neighbor_ids = []
    self_loops = []
    duplicate_neighbors = []
    over_cap = []
    zero_neighbors = []
    reverse_mismatches = []

    for layer in range(len(index.graph)):
        for node, neighbors in index.graph[layer].items():
            if node in index.deleted:
                continue

            if node in neighbors:
                self_loops.append((layer, node, neighbors[:5]))

            unique = list(dict.fromkeys(neighbors))
            if len(unique) != len(neighbors):
                duplicate_neighbors.append((layer, node, neighbors[:10]))

            if len(neighbors) > index.M:
                over_cap.append((layer, node, len(neighbors), index.M, neighbors[:10]))

            if len(neighbors) == 0:
                zero_neighbors.append((layer, node))

            for nb in neighbors:
                if nb < 0 or nb >= total_nodes:
                    invalid_neighbor_ids.append((layer, node, nb))

            for nb in neighbors:
                if nb in index.deleted:
                    invalid_neighbor_ids.append((layer, node, nb, "deleted"))

    for layer in range(len(index.graph)):
        for node, neighbors in index.graph[layer].items():
            for nb in neighbors:
                if nb >= total_nodes:
                    continue
                back = nb in index.graph[layer].get(node, [])
                if nb in index.graph[layer] and node not in index.graph[layer][nb]:
                    reverse_mismatches.append((layer, node, nb, "missing_back_edge"))

    # reachability from the entry point at each layer
    if index.entry_point is not None:
        for layer in range(len(index.graph)):
            visited = set()
            stack = [index.entry_point]
            while stack:
                current = stack.pop()
                if current in visited or current in index.deleted:
                    continue
                visited.add(current)
                for nb in index.graph[layer].get(current, []):
                    if nb not in visited and nb not in index.deleted:
                        stack.append(nb)
            reachable = len(visited)
            total_nodes_in_layer = len(index.graph[layer])
            unreachable = total_nodes_in_layer - reachable
            percent = (unreachable / total_nodes_in_layer * 100.0) if total_nodes_in_layer else 0.0
            print(f"layer {layer}: reachable={reachable}, unreachable={unreachable}, percent_unreachable={percent:.2f}%")

            if unreachable > 0:
                issues.append(f"layer {layer} unreachable nodes: {unreachable}")

    # connected components at layer 0
    layer0 = index.graph[0] if index.graph else {}
    seen = set()
    comps = []
    for node in list(layer0.keys()):
        if node in seen:
            continue
        stack = [node]
        comp = []
        while stack:
            current = stack.pop()
            if current in seen:
                continue
            seen.add(current)
            comp.append(current)
            for nb in layer0.get(current, []):
                if nb not in seen:
                    stack.append(nb)
        if comp:
            comps.append(comp)
    print(f"layer0_components={len(comps)}")
    for comp in comps[:5]:
        print(f"  component_size={len(comp)}, example_nodes={comp[:10]}")

    # final issue summary
    if index.entry_point is None or index.entry_point in index.deleted:
        issues.append(f"bad_entry_point: entry_point={index.entry_point}, max_level={index.max_level}")

    if index.max_level is None or index.max_level >= len(index.graph):
        issues.append(f"max_level_inconsistent: max_level={index.max_level}, graph_layers={len(index.graph)}")

    if self_loops:
        issues.append(f"self_loops={len(self_loops)}; example={self_loops[:max_examples]}")
    if duplicate_neighbors:
        issues.append(f"duplicate_neighbors={len(duplicate_neighbors)}; example={duplicate_neighbors[:max_examples]}")
    if over_cap:
        issues.append(f"neighbors_exceed_M={len(over_cap)}; example={over_cap[:max_examples]}")
    if zero_neighbors:
        issues.append(f"zero_neighbors={len(zero_neighbors)}; example={zero_neighbors[:max_examples]}")
    if invalid_neighbor_ids:
        issues.append(f"invalid_neighbor_ids={len(invalid_neighbor_ids)}; example={invalid_neighbor_ids[:max_examples]}")
    if reverse_mismatches:
        issues.append(f"reverse_edges_missing={len(reverse_mismatches)}; example={reverse_mismatches[:max_examples]}")

    print("\nIssue summary:")
    if not issues:
        print("No structural issues found.")
    else:
        for item in issues:
            print(" -", item)

    return {
        "entry_point": index.entry_point,
        "max_level": index.max_level,
        "layer_counts": layer_counts,
        "self_loops": self_loops[:max_examples],
        "duplicate_neighbors": duplicate_neighbors[:max_examples],
        "over_cap": over_cap[:max_examples],
        "zero_neighbors": zero_neighbors[:max_examples],
        "invalid_neighbor_ids": invalid_neighbor_ids[:max_examples],
        "reverse_mismatches": reverse_mismatches[:max_examples],
        "issues": issues,
    }


# Use this on the graph that was built and reported as invalid.
# It logs the exact failure mode without changing any HNSW parameters.
# Example:
# diagnose_hnsw_graph(hnsw)
# diagnose_hnsw_graph(hnsw_10k)

In [21]:
def build_hnsw(vectors, vector_ids, M=8, ef_construction=50, ef_search=20, seed=42):
    index = HNSWIndex(M=M, ef_construction=ef_construction, ef_search=ef_search, seed=seed)
    start = time.perf_counter()
    for i in range(len(vectors)):
        index.insert(vectors[i], int(vector_ids[i]))
    build_time = time.perf_counter() - start
    return index, build_time


def benchmark_hnsw(index, vectors, vector_ids, queries, ef_search, k=10):
    index.ef_search = int(ef_search)
    recalls = []
    latencies = []

    for q in queries:
        exact = exact_search(q, vectors, vector_ids, k=k)
        start = time.perf_counter()
        approx = index.search(q, k=k)
        latency_ms = (time.perf_counter() - start) * 1000
        latencies.append(latency_ms)
        recalls.append(recall_at_k(exact, approx, k=k))

    return {
        "recall_at_10": float(np.mean(recalls)),
        "avg_latency_ms": float(np.mean(latencies)),
        "median_latency_ms": float(np.median(latencies)),
        "min_latency_ms": float(np.min(latencies)),
        "max_latency_ms": float(np.max(latencies)),
    }


def benchmark_exact(vectors, vector_ids, queries, k=10):
    latencies = []
    for q in queries:
        start = time.perf_counter()
        exact_search(q, vectors, vector_ids, k=k)
        latencies.append((time.perf_counter() - start) * 1000)
    return {
        "avg_latency_ms": float(np.mean(latencies)),
        "median_latency_ms": float(np.median(latencies)),
        "min_latency_ms": float(np.min(latencies)),
        "max_latency_ms": float(np.max(latencies)),
    }

print("Benchmark helpers ready.")


Benchmark helpers ready.


In [22]:
DATASET_SIZES = [1000, 10000]
QUERY_COUNT = 100
K = 10
EF_SEARCH_VALUES = [10, 20, 50, 100]
M = 8
EF_CONSTRUCTION = 50
print(f"Dataset sizes: {DATASET_SIZES}")
print(f"Query count: {QUERY_COUNT}")
print(f"ef_search sweep: {EF_SEARCH_VALUES}")
print(f"Chosen config: M={M}, ef_construction={EF_CONSTRUCTION}")


Dataset sizes: [1000, 10000]
Query count: 100
ef_search sweep: [10, 20, 50, 100]
Chosen config: M=8, ef_construction=50


In [23]:
results_rows = []

for size in DATASET_SIZES:
    vectors = embeddings[:size]
    vector_ids = ids[:size]
    queries = vectors[:QUERY_COUNT]

    exact_metrics = benchmark_exact(vectors, vector_ids, queries, k=K)
    hnsw, build_time = build_hnsw(vectors, vector_ids, M=M, ef_construction=EF_CONSTRUCTION, ef_search=20)
    validation = hnsw.validate_graph()

    print(f"\n=== Dataset {size} ===")
    print(f"Build time: {build_time:.3f}s")
    print(f"Graph valid: {validation['valid']}")
    print(f"Entry point: {validation['entry_point']}")
    print(f"Avg neighbors: {validation['avg_neighbors']:.2f}")
    print(f"Max neighbors: {validation['max_neighbors']}")
    print(f"Exact avg latency: {exact_metrics['avg_latency_ms']:.3f} ms")

    for ef in EF_SEARCH_VALUES:
        metrics = benchmark_hnsw(hnsw, vectors, vector_ids, queries, ef_search=ef, k=K)
        row = {
            "dataset_size": size,
            "M": M,
            "ef_construction": EF_CONSTRUCTION,
            "ef_search": ef,
            "build_time_sec": build_time,
            "exact_avg_latency_ms": exact_metrics["avg_latency_ms"],
            "hnsw_avg_latency_ms": metrics["avg_latency_ms"],
            "hnsw_median_latency_ms": metrics["median_latency_ms"],
            "recall_at_10": metrics["recall_at_10"],
        }
        results_rows.append(row)

        print(
            f"ef={ef:>3} | Recall@10={metrics['recall_at_10']:.2%} | "
            f"HNSW avg={metrics['avg_latency_ms']:.3f} ms | "
            f"Exact avg={exact_metrics['avg_latency_ms']:.3f} ms"
        )

results_df = pd.DataFrame(results_rows)
results_df[[
    "dataset_size", "M", "ef_construction", "ef_search", "build_time_sec",
    "exact_avg_latency_ms", "hnsw_avg_latency_ms", "hnsw_median_latency_ms", "recall_at_10"
]]



=== Dataset 1000 ===
Build time: 1.984s
Graph valid: False
Entry point: 150
Avg neighbors: 7.99
Max neighbors: 8
Exact avg latency: 0.122 ms
ef= 10 | Recall@10=67.50% | HNSW avg=0.308 ms | Exact avg=0.122 ms
ef= 20 | Recall@10=72.50% | HNSW avg=0.517 ms | Exact avg=0.122 ms
ef= 50 | Recall@10=78.50% | HNSW avg=1.802 ms | Exact avg=0.122 ms
ef=100 | Recall@10=80.60% | HNSW avg=3.312 ms | Exact avg=0.122 ms

=== Dataset 10000 ===
Build time: 26.068s
Graph valid: False
Entry point: 9327
Avg neighbors: 7.99
Max neighbors: 8
Exact avg latency: 0.811 ms
ef= 10 | Recall@10=31.80% | HNSW avg=0.570 ms | Exact avg=0.811 ms
ef= 20 | Recall@10=37.70% | HNSW avg=0.774 ms | Exact avg=0.811 ms
ef= 50 | Recall@10=43.20% | HNSW avg=1.714 ms | Exact avg=0.811 ms
ef=100 | Recall@10=45.00% | HNSW avg=3.615 ms | Exact avg=0.811 ms


,dataset_size,M,ef_construction,ef_search,build_time_sec,exact_avg_latency_ms,hnsw_avg_latency_ms,hnsw_median_latency_ms,recall_at_10
0,1000,8,50,10,1.983781,0.122007,0.307838,0.28615,0.675
1,1000,8,50,20,1.983781,0.122007,0.517328,0.50480,0.725
2,1000,8,50,50,1.983781,0.122007,1.802178,1.70165,0.785
3,1000,8,50,100,1.983781,0.122007,3.311502,3.70050,0.806
4,10000,8,50,10,26.068337,0.810926,0.569619,0.51595,0.318
5,10000,8,50,20,26.068337,0.810926,0.773798,0.71640,0.377
6,10000,8,50,50,26.068337,0.810926,1.714345,1.67990,0.432
7,10000,8,50,100,26.068337,0.810926,3.614896,3.75025,0.450


In [24]:
RUN_FULL_500 = False

if RUN_FULL_500:
    query_count = 500
    full_vectors = embeddings
    full_ids = ids
    queries = full_vectors[:query_count]
    full_hnsw, full_build_time = build_hnsw(full_vectors, full_ids, M=M, ef_construction=EF_CONSTRUCTION, ef_search=50)
    full_exact = benchmark_exact(full_vectors, full_ids, queries, k=K)
    full_metrics = benchmark_hnsw(full_hnsw, full_vectors, full_ids, queries, ef_search=100, k=K)

    print(f"Full dataset size: {full_vectors.shape[0]}")
    print(f"Full build time: {full_build_time:.3f}s")
    print(f"Exact avg latency: {full_exact['avg_latency_ms']:.3f} ms")
    print(f"HNSW avg latency: {full_metrics['avg_latency_ms']:.3f} ms")
    print(f"Recall@10: {full_metrics['recall_at_10']:.2%}")
else:
    print("Full 500-query evaluation is disabled by default to avoid heavy runs. Set RUN_FULL_500 = True when ready.")


Full 500-query evaluation is disabled by default to avoid heavy runs. Set RUN_FULL_500 = True when ready.


In [25]:
# Validate a small graph to make sure the implementation is consistent before using it interactively.
small_hnsw, _ = build_hnsw(embeddings[:1000], ids[:1000], M=M, ef_construction=EF_CONSTRUCTION, ef_search=20)
small_report = small_hnsw.validate_graph()
print("Small validation report:")
for key in ["valid", "node_count", "active_nodes", "entry_point", "layers", "avg_neighbors", "max_neighbors"]:
    print(f"{key}: {small_report[key]}")
print(f"Issues: {small_report['issues']}")


Small validation report:
valid: False
node_count: 1000
active_nodes: 1000
entry_point: 150
layers: 10
avg_neighbors: 7.9871350816427515
max_neighbors: 8
Issues: ['Layer-0 reachability missing for nodes: [0, 9, 38, 40, 97, 167, 199, 202, 218, 225]']


In [26]:
def query_hnsw_from_text(index, text, k=10):
    query_vec = embed_text(text)
    hits = index.search(query_vec, k=k)
    return hits

print("Interactive search helper ready.")
print("Example: query_hnsw_from_text(hnsw, 'technology and market growth', k=5)")


Interactive search helper ready.
Example: query_hnsw_from_text(hnsw, 'technology and market growth', k=5)


In [27]:
chosen = results_df.loc[results_df["recall_at_10"].idxmax()].copy()
print("FINAL RESULTS")
print("============")
print(f"HNSW build time: {chosen['build_time_sec']:.3f} seconds")
print(f"Exact average latency: {chosen['exact_avg_latency_ms']:.3f} ms")
print(f"HNSW average latency: {chosen['hnsw_avg_latency_ms']:.3f} ms")
print(f"Recall@10: {chosen['recall_at_10']:.2%}")
print(f"Speedup: {(chosen['exact_avg_latency_ms'] / chosen['hnsw_avg_latency_ms']):.2f}x")
print(f"Chosen M: {chosen['M']}")
print(f"Chosen ef_construction: {chosen['ef_construction']}")
print(f"Chosen ef_search: {chosen['ef_search']}")
print(f"Number of vectors: {int(chosen['dataset_size'])}")


FINAL RESULTS
HNSW build time: 1.984 seconds
Exact average latency: 0.122 ms
HNSW average latency: 3.312 ms
Recall@10: 80.60%
Speedup: 0.04x
Chosen M: 8.0
Chosen ef_construction: 50.0
Chosen ef_search: 100.0
Number of vectors: 1000
